<a href="https://colab.research.google.com/github/fboldt/aulas-am-bsi/blob/main/aula12b%20pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [154]:
from sklearn.datasets import load_wine
import numpy as np

X, y = load_wine(return_X_y=True)
print(X.shape)
unique_labels, counts = np.unique(y, return_counts=True)
print("Label counts:")
for label, count in zip(unique_labels, counts):
    print(f"Label {label}: {count}")

(178, 13)
Label counts:
Label 0: 59
Label 1: 71
Label 2: 48


In [155]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [156]:
from sklearn.neighbors import KNeighborsClassifier
model = KNeighborsClassifier()

# E quanto ao preprocessamento?

In [157]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

#### ERRADO!!!! ####
## NÃO FAÇA ASSIM ##
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X) ### ERRADO!

param_grid = {
    "n_neighbors": range(1, 15, 2),
    "metric": ["euclidean", "manhattan", "chebyshev"]
}
splitter = StratifiedKFold(n_splits=5, shuffle=True)
grid_search = GridSearchCV(model, param_grid, cv=splitter, verbose=1)
scores = cross_val_score(grid_search, X_scaled, y, cv=splitter)
print(f"Acurácia final: {scores.mean()}")

Fitting 5 folds for each of 21 candidates, totalling 105 fits
Fitting 5 folds for each of 21 candidates, totalling 105 fits
Fitting 5 folds for each of 21 candidates, totalling 105 fits
Fitting 5 folds for each of 21 candidates, totalling 105 fits
Fitting 5 folds for each of 21 candidates, totalling 105 fits
Acurácia final: 0.9774603174603176


# Pipelines

In [164]:
from sklearn.base import BaseEstimator, ClassifierMixin

class Pipeline(BaseEstimator, ClassifierMixin):
  def __init__(self, preprocessing, classifier):
    self.preprocessing = preprocessing
    self.classifier = classifier
  def fit(self, X, y):
    X = self.preprocessing.fit_transform(X)
    self.classifier.fit(X, y)
    return self
  def predict(self, X):
    X = self.preprocessing.transform(X)
    return self.classifier.predict(X)

scaler = StandardScaler()
knn = KNeighborsClassifier()

model = Pipeline(scaler, knn)
model.fit(X_train, y_train)
print(f"Acurácia final: {model.score(X_test, y_test)}")

Acurácia final: 0.9722222222222222


In [173]:
splitter = StratifiedKFold(n_splits=5, shuffle=True)
model = Pipeline(scaler, knn)
scores = cross_val_score(model, X, y, cv=splitter)
print(f"Acurácia final: {scores.mean()}")

Acurácia final: 0.972063492063492


In [174]:
from sklearn.pipeline import Pipeline

model = Pipeline([
  ("scaler", StandardScaler()),
  ("knn", KNeighborsClassifier())
])
model.fit(X_train, y_train)
print(f"Acurácia final: {model.score(X_test, y_test)}")

Acurácia final: 0.9722222222222222


In [180]:
scores = cross_val_score(model, X, y, cv=splitter)
print(f"Acurácia final: {scores.mean()}")

Acurácia final: 0.9715873015873017


# Tá! Mas como encontrar o hiperparâmetros disso?

In [183]:
model = Pipeline([
  ("scaler", StandardScaler()),
  ("knn", KNeighborsClassifier())
])
param_grid = {
    "scaler__with_mean": [True, False],
    "scaler__with_std": [True, False],
    "knn__n_neighbors": range(1, 15, 2),
    "knn__metric": ["euclidean", "manhattan", "chebyshev"]
}
splitter = StratifiedKFold(n_splits=5, shuffle=True)
grid_search = GridSearchCV(model, param_grid, cv=splitter, verbose=1)
scores = cross_val_score(grid_search, X_scaled, y, cv=splitter)
print(f"Acurácia final: {scores.mean()}")

Fitting 5 folds for each of 84 candidates, totalling 420 fits
Fitting 5 folds for each of 84 candidates, totalling 420 fits
Fitting 5 folds for each of 84 candidates, totalling 420 fits
Fitting 5 folds for each of 84 candidates, totalling 420 fits
Fitting 5 folds for each of 84 candidates, totalling 420 fits
Acurácia final: 0.9609523809523809


In [184]:
model = Pipeline([
  ("scaler", StandardScaler()),
  ("knn", KNeighborsClassifier())
])
param_grid = {
    "scaler__with_mean": [True, False],
    "scaler__with_std": [True, False],
    "knn__n_neighbors": range(1, 5, 2),
    "knn__metric": ["euclidean", "manhattan"]
}
splitter = StratifiedKFold(n_splits=2, shuffle=True)
grid_search = GridSearchCV(model, param_grid, cv=splitter, verbose=3)
scores = cross_val_score(grid_search, X_scaled, y, cv=splitter)
print(f"Acurácia final: {scores.mean()}")

Fitting 2 folds for each of 16 candidates, totalling 32 fits
[CV 1/2] END knn__metric=euclidean, knn__n_neighbors=1, scaler__with_mean=True, scaler__with_std=True;, score=0.911 total time=   0.0s
[CV 2/2] END knn__metric=euclidean, knn__n_neighbors=1, scaler__with_mean=True, scaler__with_std=True;, score=0.932 total time=   0.0s
[CV 1/2] END knn__metric=euclidean, knn__n_neighbors=1, scaler__with_mean=True, scaler__with_std=False;, score=0.911 total time=   0.0s
[CV 2/2] END knn__metric=euclidean, knn__n_neighbors=1, scaler__with_mean=True, scaler__with_std=False;, score=0.932 total time=   0.0s
[CV 1/2] END knn__metric=euclidean, knn__n_neighbors=1, scaler__with_mean=False, scaler__with_std=True;, score=0.911 total time=   0.0s
[CV 2/2] END knn__metric=euclidean, knn__n_neighbors=1, scaler__with_mean=False, scaler__with_std=True;, score=0.932 total time=   0.0s
[CV 1/2] END knn__metric=euclidean, knn__n_neighbors=1, scaler__with_mean=False, scaler__with_std=False;, score=0.911 total t